# F4 — Retrieval-Augmented Generation · Hands-On Lab Notebook

**AI Engineering Specialist Track · Foundation Module F4**

Three labs, ~65 minutes, fully runnable offline:

1. **Chunking & embeddings** — split a knowledge base into chunks, embed them into vectors, and inspect the vector space.
2. **Vector store & retrieval** — build an in-memory vector store, do similarity search, and compare chunking strategies with `hit@k`.
3. **The full RAG pipeline** — retrieve → augment → generate a *grounded, cited* answer, and measure groundedness against a no-retrieval baseline.

---

### A note on how this notebook runs

The labs teach the **real** RAG pipeline — chunking, embeddings, cosine retrieval,
grounded generation with citations, and retrieval/faithfulness evaluation.

To guarantee it runs **with zero errors and zero warnings on any machine**, it uses
small, self-contained stand-ins instead of the heavy stack: a pure-NumPy TF-IDF embedder
in place of a downloaded dense-embedding model, and an in-memory NumPy vector store in
place of FAISS/Chroma. Every step maps 1:1 to the real components — the *Case Analysis
Solutions* doc shows the exact production equivalent for each lab. In a real system you
swap the embedder and the store; the pipeline around them is identical.

## Setup

Only NumPy, Pandas, and Matplotlib — nothing that downloads models or needs an API key.

In [ ]:
# Clean, quiet environment ----------------------------------------------------
import os, warnings, logging, re, math
from dataclasses import dataclass, field
from typing import Callable

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
logging.getLogger("matplotlib").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
print("Environment ready.")
print("NumPy:", np.__version__, "| Pandas:", pd.__version__)

## The knowledge base

A small telecom support knowledge base — the kind of corpus the **RAG node** from
F3's triage graph would search. Each article has an id, a title, and a body. This is the
ground truth the model must answer *from* — not from its own memory.

In [ ]:
KB_ARTICLES = [
    {"id": "KB01", "title": "Resetting your account password",
     "body": ("If you are locked out of your account, go to the login page and select "
              "Forgot Password. A reset link is emailed to your registered address and is "
              "valid for 30 minutes. If the email does not arrive, check spam, then request "
              "again after five minutes. Repeated failures may mean the email on file is "
              "outdated; contact support to verify your identity and update it.")},
    {"id": "KB02", "title": "Understanding charges on your bill",
     "body": ("Your monthly bill lists the plan fee, taxes, and any one-time charges such as "
              "device instalments or roaming. A charge you do not recognise is often a "
              "pro-rated adjustment from a mid-cycle plan change. Disputed charges can be "
              "raised within 60 days; approved disputes are credited to the next bill.")},
    {"id": "KB03", "title": "Troubleshooting slow or dropping internet",
     "body": ("Intermittent connection drops are usually caused by an overheating router or "
              "local network congestion. Power-cycle the router by unplugging it for 30 "
              "seconds. Keep it ventilated and away from other electronics. If drops persist "
              "during specific hours, congestion in your area may be the cause; report the "
              "pattern so engineering can investigate.")},
    {"id": "KB04", "title": "Activating a new SIM or device",
     "body": ("To activate a new SIM, insert it and restart the device, then dial the "
              "activation code sent by SMS. Activation completes within 15 minutes. For an "
              "eSIM, scan the QR code from your confirmation email. A device that shows no "
              "signal after activation usually needs a network settings reset.")},
    {"id": "KB05", "title": "Cancelling your service",
     "body": ("To cancel, submit a request from the account portal at least 30 days before "
              "your next billing date to avoid a further charge. Outstanding device "
              "instalments are billed in full on cancellation. A final invoice is issued "
              "within one billing cycle, and any deposit is refunded after equipment return.")},
    {"id": "KB06", "title": "Roaming and international usage",
     "body": ("Roaming is disabled by default. Enable it in the app under Plan settings before "
              "you travel. International data is billed per megabyte unless you add a travel "
              "pass, which bundles a data allowance for a fixed daily fee. Calls received "
              "while roaming may incur a charge even if you do not answer.")},
    {"id": "KB07", "title": "Upgrading or changing your plan",
     "body": ("Plan changes take effect at the start of the next billing cycle by default, or "
              "immediately if you opt in, which creates a pro-rated charge. Upgrading keeps "
              "your number and data history. Downgrading may remove add-ons such as extra "
              "data, so review your usage before switching.")},
    {"id": "KB08", "title": "Data usage and overage charges",
     "body": ("You can track data usage in real time in the app dashboard. When you reach 80 "
              "percent of your allowance the system sends an alert. Exceeding the allowance "
              "triggers overage billed per gigabyte, unless your plan throttles speed instead. "
              "Add a data booster mid-cycle to avoid overage at a lower cost.")},
]
print(f"Knowledge base: {len(KB_ARTICLES)} articles")
for a in KB_ARTICLES[:3]:
    print(f"  {a['id']}: {a['title']}")

---
# Lab 1 — Chunking & embeddings

**Goal:** split the knowledge base into retrievable **chunks**, then **embed** each chunk
into a vector. Retrieval works on chunks, not whole documents — so the chunking strategy
is a real engineering decision, not an afterthought.

### Step 1.1 — A chunker (fixed window with overlap)

We split each article body into overlapping windows of `chunk_size` words. Overlap keeps a
sentence that straddles a boundary retrievable from either side. Real equivalent:
LangChain's `RecursiveCharacterTextSplitter`.

### Step 1.2 — A deterministic embedder (TF-IDF, pure NumPy)

This maps text to a vector so similar text lands nearby. We implement TF-IDF in NumPy: a
vector over the corpus vocabulary, weighted by term frequency × inverse document frequency,
then L2-normalised so cosine similarity is a simple dot product.

> **In production** this slot is a dense embedding model — `BAAI/bge-*`, OpenAI
> `text-embedding-3`, Cohere `embed-*` — which captures *semantic* similarity, not just word
> overlap. The pipeline below is identical; only `embed()` changes.

### Step 1.3 — Sanity-check the vector space

In [ ]:
# Visualise: how similar is the query to each article (max chunk similarity per article)?
per_article = {}
for idx, c in enumerate(chunks):
    per_article[c["source_id"]] = max(per_article.get(c["source_id"], 0.0), sims[idx])
labels = list(per_article.keys())
vals = [per_article[k] for k in labels]
fig, ax = plt.subplots(figsize=(8.5, 3.4))
bars = ax.bar(labels, vals, color="#A21D52")
top = int(np.argmax(vals))
bars[top].set_color("#C2820B")
ax.set_title("Lab 1 — query-to-article similarity (max chunk sim)", fontsize=12, fontweight="bold")
ax.set_ylabel("cosine similarity")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()
print("Best-matching article is highlighted — retrieval is working.")

### Lab 1 — What to look for (discussion)

1. **Chunking is a real decision.** Too small and a chunk loses context; too large and retrieval gets imprecise and dilutes the signal. Overlap rescues sentences that straddle a boundary.
2. **Embeddings turn meaning into geometry.** Similar text → nearby vectors. Retrieval is just "find the nearest vectors to the query."
3. **This embedder is lexical (word overlap).** A real dense model captures *semantics* — "locked out" near "password reset" even without shared words. The pipeline is identical; swap `embed()`.
4. **Normalisation matters.** L2-normalising makes cosine similarity a dot product and keeps long and short chunks comparable.

---
# Lab 2 — A vector store & semantic retrieval

**Goal:** wrap the embeddings in a **vector store** that supports `add` and `search`, then
measure retrieval quality with `hit@k` — and use it to compare two chunking strategies.

### Step 2.1 — An in-memory vector store

`add` stores chunk vectors + metadata; `search` embeds the query and returns the top-k by
cosine similarity. Real equivalent: FAISS, Chroma, or pgvector — same `add`/`search` shape.

### Step 2.2 — An evaluation set and `hit@k`

A handful of queries, each labelled with the article that *should* answer it. `hit@k` is the
fraction of queries whose relevant article appears in the top-k retrieved chunks — the most
basic retrieval-quality metric.

In [ ]:
EVAL_QUERIES = [
    ("how do I reset my password",                 "KB01"),
    ("there is a charge on my bill I don't know",  "KB02"),
    ("my internet keeps dropping out",             "KB03"),
    ("how do I activate my new sim card",          "KB04"),
    ("I want to cancel my service",                "KB05"),
    ("will I be charged for using data abroad",    "KB06"),
    ("how do I upgrade to a bigger plan",          "KB07"),
    ("what happens if I go over my data",          "KB08"),
]



### Step 2.3 — Compare chunking strategies on retrieval quality

Same corpus, same queries — only the chunk size changes. This is exactly the kind of
experiment you run when tuning a real RAG system.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.6))
labels = [f"{r['chunk_size']}w/{r['overlap']}o\n({r['n_chunks']} chunks)" for r in results]
x = np.arange(len(results))
w = 0.38
ax.bar(x - w/2, [r["hit@1"] for r in results], width=w, label="hit@1", color="#A21D52")
ax.bar(x + w/2, [r["hit@3"] for r in results], width=w, label="hit@3", color="#0E7490")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_title("Lab 2 — chunking strategy vs retrieval quality", fontsize=12, fontweight="bold")
ax.set_ylabel("score")
ax.legend(frameon=False, loc="lower right")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

### Step 2.4 — Rebuild the pipeline with the winning configuration

The comparison shows 80-word chunks retrieve best on this corpus. That's the real RAG dev
loop: chunk → evaluate → re-chunk with the winner. We rebuild `chunks`, `embedder`, and
`store` with the tuned config and carry them into Lab 3.

In [ ]:
best = max(results, key=lambda r: (r["hit@3"], -r["chunk_size"]))
print(f"Best config: chunk_size={best['chunk_size']}, overlap={best['overlap']} "
      f"(hit@3={best['hit@3']})")

# Rebuild the pipeline with the winning configuration for the rest of the labs.
chunks   = build_chunks(KB_ARTICLES, chunk_size=best["chunk_size"], overlap=best["overlap"])
embedder = TfidfEmbedder().fit([c["embed_text"] for c in chunks])
store    = VectorStore(embedder).add(chunks)

tuned_score, _ = hit_at_k(store, EVAL_QUERIES, k=3)
print(f"Tuned store: {len(chunks)} chunks, hit@3 = {tuned_score:.3f} "
      f"(up from 0.62 with the 40-word default).")
print("One query still misses — 'upgrade to a bigger plan' — a lexical gap a real")
print("dense embedder would close by matching meaning, not words.")

### Lab 2 — What to look for (discussion)

1. **The store is two methods: `add` and `search`.** Everything else — FAISS, Chroma, pgvector — is that interface plus scale, persistence, and filtering.
2. **`hit@k` is the retrieval floor.** If the right article isn't in the top-k, no amount of clever prompting will produce a grounded answer. Fix retrieval before you touch the generator.
3. **Chunk size is a tunable knob.** Very large chunks (one chunk per article) can still score well on a tiny corpus but hurt precision and cost on a real one. Measure, don't guess.
4. **This is offline evaluation.** A labelled query→document set lets you compare strategies objectively — the same discipline as F2's prompt eval, applied to retrieval.

---
# Lab 3 — The full RAG pipeline: grounded, cited answers

**Goal:** assemble **retrieve → augment → generate**, producing an answer grounded in the
retrieved chunks **with a citation** — then measure *groundedness* and contrast it with a
no-retrieval baseline that hallucinates.

### Step 3.1 — A grounded generator

Given a query and retrieved chunks, the generator answers using **only** the retrieved
context — here, extractively (the most query-relevant sentence from the top chunk) with a
citation. Real equivalent: an LLM prompted with "answer using only the context below, and
cite your source." 

In [ ]:
_sent_re = re.compile(r"(?<=[.!?])\s+")
BODY_BY_ID = {a["id"]: a["body"] for a in KB_ARTICLES}

def _split_sentences(text):
    return [s.strip() for s in _sent_re.split(text) if s.strip()]

def grounded_generate(query, retrieved, embedder):
    """Answer using ONLY the retrieved sources. Extractive + cited + deterministic.
    Pulls a COMPLETE sentence from the retrieved chunk's source article (not the possibly
    truncated chunk window), so answers are never sentence fragments."""
    if not retrieved:
        return {"answer": "I don't have information on that.", "citation": None, "grounded": True}
    qv = embedder.embed(query)
    best_sent, best_src, best_sim = None, None, -1.0
    seen = []
    for r in retrieved:                       # consider each distinct retrieved source
        if r["source_id"] in seen:
            continue
        seen.append(r["source_id"])
        for sent in _split_sentences(BODY_BY_ID[r["source_id"]]):
            sim = float(np.dot(embedder.embed(sent), qv))
            if sim > best_sim:
                best_sent, best_src, best_sim = sent, r, sim
    return {"answer": best_sent, "citation": f"{best_src['source_id']} — {best_src['title']}",
            "grounded": True}

def rag_answer(query, store, embedder, k=3):
    retrieved = store.search(query, k=k)
    out = grounded_generate(query, retrieved, embedder)
    out["retrieved"] = [f"{r['source_id']}({r['score']:.2f})" for r in retrieved]
    return out

example = rag_answer("will I be charged for receiving calls while abroad", store, embedder)
print("Q: will I be charged for receiving calls while abroad\n")
print("A:", example["answer"])
print("\nCitation:", example["citation"])
print("Retrieved:", example["retrieved"])

### Step 3.2 — The contrast: RAG vs no-retrieval

The no-retrieval baseline answers from a fixed "prior" with no access to the knowledge base
— the kind of plausible-but-unsourced answer that causes hallucination in production.

In [ ]:
# A canned no-context answer: fluent, generic, and NOT grounded in the KB.
NO_CONTEXT_ANSWER = ("You can usually manage that in your account settings; "
                     "most providers handle it automatically within a day or two.")



In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))
x = np.arange(len(df_ground))
w = 0.38
ax.bar(x - w/2, df_ground["rag_grounded"], width=w, label="RAG (grounded)", color="#A21D52")
ax.bar(x + w/2, df_ground["no_ctx_grounded"], width=w, label="No-context baseline", color="#9AA5AD")
ax.set_xticks(x)
ax.set_xticklabels([g for _, g in EVAL_QUERIES], rotation=0)
ax.set_ylim(0, 1.05)
ax.set_ylabel("groundedness")
ax.set_title("Lab 3 — RAG answers are grounded; the baseline is not", fontsize=12, fontweight="bold")
ax.legend(frameon=False, loc="upper right")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

### Step 3.3 — Inspect a few grounded answers with citations

In [ ]:
for q in ["how do I activate my new sim card",
          "what happens if I exceed my data allowance",
          "how long is the password reset link valid"]:
    out = rag_answer(q, store, embedder, k=3)
    print(f"Q: {q}")
    print(f"A: {out['answer']}")
    print(f"   [source: {out['citation']}]\n")

### Lab 3 — What to look for (discussion)

1. **Grounding is the whole point of RAG.** The RAG answers score high on groundedness; the no-context baseline scores near zero. That gap *is* the value — answers tied to a source instead of invented.
2. **Citations make it auditable.** Every answer points to the article it came from. In a regulated setting, an answer without a citation is a liability.
3. **Retrieval quality caps answer quality.** Lab 2's `hit@k` is the ceiling — if retrieval misses, the generator has nothing true to say. Garbage in, hallucination out.
4. **Faithfulness is measurable.** Groundedness here is a simple token-overlap proxy; production tools (RAGAS, LangSmith evals) score faithfulness and context relevance with an LLM judge. Same idea, richer signal.

**Next:** F5 — Fine-tuning & model adaptation. When prompting and retrieval aren't enough, you adapt the model itself.

---
# F4 Lab — Wrap-up checklist

- [ ] Why does chunk size trade off context against retrieval precision?
- [ ] What does an embedding actually represent, and why normalise it?
- [ ] What two methods define a vector store, and what do FAISS/Chroma add on top?
- [ ] What does `hit@k` measure, and why fix retrieval before prompting?
- [ ] In RAG, what are the three stages, and which one caps the others?
- [ ] What is groundedness, and how does it expose a hallucinating baseline?
- [ ] Where would a real dense embedding model change the results vs this lexical one?